In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import logging
import csv
import json
from datetime import datetime

# Setup Logging
logging.basicConfig(
    filename='wiki_scraping_log.log', 
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    force=True
)

def web_scrapping_wiki(url, article_num):
    """
    Visits the Wikipedia URL, grabs the title and the paragraph text.
    """
    # Using your MRU identity to be polite to Wikipedia's servers
    headers = {'User-Agent': 'MRU Data Science Student Project (Contact: data_science_student@mtroyal.ca)'}
    
    # Polite delay so Wikipedia doesn't block us
    time.sleep(2)
    
    try:
        logging.info(f"Starting request for: {url}")
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.content, 'html.parser')

        dict_data = {}
        dict_data['Article_ID'] = f'WIKI_A{article_num}'
        dict_data['Source_ID'] = 'wikipedia.org'
        
        # Wikipedia stores its main title in an h1 tag with id 'firstHeading'
        title_element = soup.find('h1', id='firstHeading')
        dict_data['Title'] = title_element.text if title_element else "Unknown Title"
        
        dict_data['Publish_Date'] = "N/A" # Wikipedia articles are continuously updated
        dict_data['URL'] = url             
        
        # Grab all standard paragraphs and join them into one block of text
        paragraphs = soup.find_all("p")
        dict_data['Article_Text'] = " ".join(p.get_text(strip=True) for p in paragraphs if p.get_text(strip=True))

        logging.info(f"Successfully scraped: {url}")
        return dict_data

    except Exception as e:
        logging.error(f"Error scraping {url}: {e}")
        return None

def analyze(data):
    target_keywords = {
    "Russia": ["Russia", "Russian", "Moscow", "AIN", "Neutral Athlete"],
    "Belarus": ["Belarus", "Belarusian", "Minsk"],
    "Israel": ["Israel", "Israeli", "Tel Aviv"],
    "China": ["China", "Chinese", "Beijing"],
    "North Korea": ["North Korea", "DPRK", "Korean"],
    "Iran": ["Iran", "Iranian", "Tehran"],
    "Ukraine": ["Ukraine", "Ukrainian", "Kyiv"]
}
    political_keywords = ["war", "invasion", "sanction", "protest", "ban", "neutrality", "conflict"]
    
    url = data['URL']
    title = data['Title']
    content_to_check = (url + " " + title).lower()

    found_countries = ''
    
    for country, keywords in target_keywords.items():
        if any(kw.lower() in content_to_check for kw in keywords):
            found_countries += f'{country},'

    if found_countries == '': 
        return None
    
    text = data['Article_Text'].lower()

    mention_count = sum(text.count(kw.lower()) for kw in target_keywords.values() for kw in kw)
    has_political_context = sum(text.count(p_kw.lower()) for p_kw in political_keywords)

    if mention_count >= 2 and has_political_context:
        framing_analysis = {}
        framing_analysis['Article_ID'] = data['Article_ID']
        framing_analysis['Target_Nation'] = found_countries[:-1]
        framing_analysis['Frequency_of_Political_Terms'] = has_political_context
        return framing_analysis
    
    return None 

def update_acquisition_log(domain, records_count, errors_count, related_count):
    log_file = 'acquisition_log.csv'
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    with open(log_file, mode='a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([timestamp, domain, records_count, errors_count, related_count])

def save_to_json(articles, analysis):
    with open("Wiki_Articles.json", "w", encoding='utf-8') as f:
        json.dump(articles, f, indent=4, ensure_ascii=False)

    with open("Wiki_Source_ID.json", "w", encoding='utf-8') as f:
        json.dump(analysis, f, indent=4, ensure_ascii=False)

def main():
    print("Loading Phase 2 URLs...")
    # Load the JSON file you generated in Phase 2
    with open('selenium_articles_data.json', 'r', encoding='utf-8') as f:
        raw_wiki_data = json.load(f)

    success_count = 0 
    errors_count = 0 
    related_count = 0 

    articles_ = []
    framing_analysis_ = []

    print(f"Found {len(raw_wiki_data)} URLs to process. Starting Deep Crawl & Filter...")

    for index, item in enumerate(raw_wiki_data, start=1):
        # We only want to process the URLs from the Phase 2 JSON
        url = item.get('URL')
        
        if not url or "wikipedia.org" not in url:
            continue

        print(f"Processing {index}/{len(raw_wiki_data)}: {url}")
        
        # 1. Scrape the text
        result = web_scrapping_wiki(url, index)

        if result is None:
            errors_count += 1
            continue

        success_count += 1
        
        # 2. Filter using your analyze function
        analysis = analyze(result)

        if analysis is None: 
            # If it doesn't mention the countries or political words, we throw it out!
            continue

        related_count += 1
        articles_.append(result)
        framing_analysis_.append(analysis)

    # 3. Save the filtered 
    save_to_json(articles_, framing_analysis_)
    update_acquisition_log("wikipedia.org", success_count, errors_count, related_count)
    
    print(f"\nFINISHED! Out of {success_count} pages read, {related_count} passed the political keyword filter.")
            
if __name__ == "__main__":
    main()
    

Loading Phase 2 URLs...
Found 1153 URLs to process. Starting Deep Crawl & Filter...
Processing 1/1153: https://en.wikipedia.org/wiki/2026_Winter_Olympics#bodyContent
Processing 2/1153: https://en.wikipedia.org/wiki/2026_Winter_Olympics#
Processing 3/1153: https://en.wikipedia.org/wiki/2026_Winter_Olympics#Bidding_process
Processing 4/1153: https://en.wikipedia.org/wiki/2026_Winter_Olympics#Host_city_selection
Processing 5/1153: https://en.wikipedia.org/wiki/2026_Winter_Olympics#Development_and_preparations
Processing 6/1153: https://en.wikipedia.org/wiki/2026_Winter_Olympics#Venues
Processing 7/1153: https://en.wikipedia.org/wiki/2026_Winter_Olympics#Milan_cluster
Processing 8/1153: https://en.wikipedia.org/wiki/2026_Winter_Olympics#Cortina_d'Ampezzo_cluster
Processing 9/1153: https://en.wikipedia.org/wiki/2026_Winter_Olympics#Valtellina_cluster
Processing 10/1153: https://en.wikipedia.org/wiki/2026_Winter_Olympics#Val_di_Fiemme_cluster
Processing 11/1153: https://en.wikipedia.org/wiki